In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

# Project Data Dictionary

Generate the shared data dictionary for all source and team-created fields.

Unknown predictor meanings stay marked as `unknown`. No field meaning is inferred from the anonymous names.

## Setup

Use the authoritative source and the Gate 1 profile to build the dictionary.

In [2]:
def find_repo_root():
    path = Path.cwd().resolve()

    for parent in [path, *path.parents]:
        if (parent / "data" / "allstate_claims_data.csv").exists():
            return parent

    raise FileNotFoundError("Could not find data/allstate_claims_data.csv")


REPO_ROOT = find_repo_root()

DATA_PATH = REPO_ROOT / "data" / "allstate_claims_data.csv"

GATE1_DIR = (
    REPO_ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 1"
)

GATE2_DIR = (
    REPO_ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 2"
)

PROFILE_PATH = GATE1_DIR / "profile.csv"
DICTIONARY_PATH = GATE2_DIR / "data_dictionary.csv"

DICTIONARY_VERSION = "1.0.0"

ID_COL = "id"
CAT_COLS = [f"cat{i}" for i in range(1, 117)]
CONT_COLS = [f"cont{i}" for i in range(1, 15)]
TARGET_COL = "loss"

EXPECTED_SOURCE_FIELDS = (
    [ID_COL]
    + CAT_COLS
    + CONT_COLS
    + [TARGET_COL]
)

print(f"Source: {DATA_PATH.relative_to(REPO_ROOT)}")
print(f"Profile: {PROFILE_PATH.relative_to(REPO_ROOT)}")
print(f"Output: {DICTIONARY_PATH.relative_to(REPO_ROOT)}")

Source: data\allstate_claims_data.csv
Profile: notebooks\final-deliverables\September\Gate 1\profile.csv
Output: notebooks\final-deliverables\September\Gate 2\data_dictionary.csv


## Load Source and Profile

The profile provides the observed types, unique counts, ranges, and missingness from the clean Gate 1 workflow.

In [3]:
df = pd.read_csv(DATA_PATH)
profile = pd.read_csv(PROFILE_PATH)

print(f"Source shape: {df.shape}")
print(f"Profile rows: {len(profile)}")

Source shape: (188318, 132)
Profile rows: 132


In [4]:
expected_profile_fields = set(EXPECTED_SOURCE_FIELDS)
actual_profile_fields = set(profile["column"])

if expected_profile_fields != actual_profile_fields:
    missing = expected_profile_fields - actual_profile_fields
    extra = actual_profile_fields - expected_profile_fields

    raise RuntimeError(
        f"Profile fields do not match the expected schema.\n"
        f"Missing: {sorted(missing)}\n"
        f"Extra: {sorted(extra)}"
    )

print("Profile schema: PASS")

Profile schema: PASS


## Source Field Rules

Apply the documented roles and handling rules consistently by field family.

In [5]:
def source_metadata(field):
    if field == "id":
        return {
            "approved_role": "identifier",
            "unit_or_scale": "claim identifier",
            "documented_meaning": "identifier for an insurance claim",
            "quality_limitation": (
                "Identifier values have no documented analytical meaning."
            ),
            "september_handling": (
                "Use for integrity checks only. Exclude from "
                "distribution and target-effect interpretation."
            ),
            "meaning_status": "documented",
        }

    if field in CAT_COLS:
        return {
            "approved_role": "anonymous nominal categorical predictor",
            "unit_or_scale": "nominal categorical",
            "documented_meaning": "unknown",
            "quality_limitation": (
                "Anonymous predictor; semantic meaning is not documented."
            ),
            "september_handling": (
                "Treat as an unordered nominal categorical predictor. "
                "Do not infer a meaning or ordering."
            ),
            "meaning_status": "unresolved",
        }

    if field in CONT_COLS:
        return {
            "approved_role": "anonymous continuous predictor",
            "unit_or_scale": "scaled between 0 and 1",
            "documented_meaning": "unknown",
            "quality_limitation": (
                "Original unit and transformation are not documented."
            ),
            "september_handling": (
                "Treat as a continuous predictor on the delivered 0-to-1 scale. "
                "Do not infer the original unit or transformation."
            ),
            "meaning_status": "unresolved",
        }

    if field == "loss":
        return {
            "approved_role": "regression target",
            "unit_or_scale": "USD (dollars)",
            "documented_meaning": "final paid claim amount",
            "quality_limitation": (
                "Closed claims without payment are excluded from "
                "the delivered dataset. Individual inclusion reasons "
                "are not documented."
            ),
            "september_handling": (
                "Retain as the regression target. Use original loss "
                "units for reporting and model evaluation (MAE). "
                "Do not infer causation or individual reserve amounts."
            ),
            "meaning_status": "documented",
        }

    raise ValueError(f"Unexpected field: {field}")

## Observed Values

Record numeric ranges for numeric fields and exact observed levels for categorical fields.

In [6]:
def get_category_levels(series):
    levels = sorted(series.dropna().astype(str).unique().tolist())

    # JSON keeps the levels machine-readable inside the CSV cell.
    return json.dumps(levels)

## Source Fields

Build one dictionary row for each of the 132 delivered fields.

In [7]:
profile_lookup = profile.set_index("column")

source_rows = []

for field in EXPECTED_SOURCE_FIELDS:
    p = profile_lookup.loc[field]
    metadata = source_metadata(field)

    if field in CAT_COLS:
        category_levels = get_category_levels(df[field])
        observed_min = None
        observed_max = None
    else:
        category_levels = None
        observed_min = p["min"]
        observed_max = p["max"]

    source_rows.append({
        "field_name": field,
        "field_origin": "source",
        "source_name": field,
        "approved_role": metadata["approved_role"],
        "observed_dtype": p["source_dtype"],
        "observed_min": observed_min,
        "observed_max": observed_max,
        "category_levels": category_levels,
        "n_unique": int(p["n_unique"]),
        "missing_count": int(p["missing"]),
        "missing_pct": float(p["missing"]) / len(df) * 100,
        "unit_or_scale": metadata["unit_or_scale"],
        "documented_meaning": metadata["documented_meaning"],
        "quality_limitation": metadata["quality_limitation"],
        "september_handling": metadata["september_handling"],
        "meaning_status": metadata["meaning_status"],
        "purpose": None,
        "inputs": None,
        "exact_derivation": None,
        "allowed_values": None,
        "field_version": None,
        "owner": None,
        "dictionary_version": DICTIONARY_VERSION,
    })

source_dictionary = pd.DataFrame(source_rows)

print(f"Source fields: {len(source_dictionary)}")
source_dictionary.head()

Source fields: 132


,field_name,field_origin,source_name,approved_role,observed_dtype,observed_min,observed_max,category_levels,n_unique,missing_count,...,quality_limitation,september_handling,meaning_status,purpose,inputs,exact_derivation,allowed_values,field_version,owner,dictionary_version
0,id,source,id,identifier,int64,1.0,587633.0,None,188318,0,...,Identifier values have no documented analytica...,Use for integrity checks only. Exclude from di...,documented,None,None,None,None,None,None,1.0.0
1,cat1,source,cat1,anonymous nominal categorical predictor,object,NaN,NaN,"[""A"", ""B""]",2,0,...,Anonymous predictor; semantic meaning is not d...,Treat as an unordered nominal categorical pred...,unresolved,None,None,None,None,None,None,1.0.0
2,cat2,source,cat2,anonymous nominal categorical predictor,object,NaN,NaN,"[""A"", ""B""]",2,0,...,Anonymous predictor; semantic meaning is not d...,Treat as an unordered nominal categorical pred...,unresolved,None,None,None,None,None,None,1.0.0
3,cat3,source,cat3,anonymous nominal categorical predictor,object,NaN,NaN,"[""A"", ""B""]",2,0,...,Anonymous predictor; semantic meaning is not d...,Treat as an unordered nominal categorical pred...,unresolved,None,None,None,None,None,None,1.0.0
4,cat4,source,cat4,anonymous nominal categorical predictor,object,NaN,NaN,"[""A"", ""B""]",2,0,...,Anonymous predictor; semantic meaning is not d...,Treat as an unordered nominal categorical pred...,unresolved,None,None,None,None,None,None,1.0.0


## Source Dictionary Check

Make sure all 132 source fields are present once and in the documented order.

In [8]:
source_dictionary_check = (
    len(source_dictionary) == 132
    and source_dictionary["field_name"].tolist() == EXPECTED_SOURCE_FIELDS
    and source_dictionary["field_name"].is_unique
)

print(f"132 source fields: {len(source_dictionary) == 132}")
print(
    "Correct field order:",
    source_dictionary["field_name"].tolist() == EXPECTED_SOURCE_FIELDS
)
print(
    "Unique field names:",
    source_dictionary["field_name"].is_unique
)

if not source_dictionary_check:
    raise RuntimeError("Source dictionary validation failed.")

132 source fields: True
Correct field order: True
Unique field names: True


## Derived Fields

Team-created fields are documented separately with their purpose, inputs, exact derivation, allowed values, version, and owner.

Only finalized fields should be added here.

In [9]:
DERIVED_FIELDS = [
    {
        "field_name": "log1p_loss",
        "field_origin": "derived",
        "source_name": None,
        "approved_role": "derived target",
        "observed_dtype": "float64",
        "observed_min": None,
        "observed_max": None,
        "category_levels": None,
        "n_unique": None,
        "missing_count": None,
        "missing_pct": None,
        "unit_or_scale": "log1p transformation of loss",
        "documented_meaning": "log-transformed total paid claim amount",
        "quality_limitation": (
            "Log-transformed values are not dollar amounts and "
            "cannot be interpreted directly as claim costs."
        ),
        "september_handling": (
            "Use for exploratory analysis only. Retain original "
            "loss values for reporting and model evaluation (MAE)."
        ),
        "meaning_status": "documented",
        "purpose": (
            "Support exploratory analysis of the loss distribution "
            "using a log-transformed scale."
        ),
        "inputs": "loss",
        "exact_derivation": "log1p_loss = ln(1 + loss)",
        "allowed_values": "[0, infinity)",
        "field_version": "1.0",
        "owner": "Liam",
        "dictionary_version": DICTIONARY_VERSION,
    },
]

## Calculate Derived Field Profiles

Where possible, calculate observed metadata for finalized derived fields from their documented derivations.

In [10]:
derived_df = df.copy()

derived_df["log1p_loss"] = np.log1p(
    derived_df["loss"]
)

In [11]:
derived_rows = []

for metadata in DERIVED_FIELDS:
    field = metadata["field_name"]

    row = metadata.copy()

    if field in derived_df.columns:
        series = derived_df[field]

        row["observed_dtype"] = str(series.dtype)
        row["n_unique"] = int(series.nunique(dropna=False))
        row["missing_count"] = int(series.isna().sum())
        row["missing_pct"] = float(series.isna().mean() * 100)

        if pd.api.types.is_numeric_dtype(series):
            row["observed_min"] = float(series.min())
            row["observed_max"] = float(series.max())

    derived_rows.append(row)

derived_dictionary = pd.DataFrame(derived_rows)

derived_dictionary

,field_name,field_origin,source_name,approved_role,observed_dtype,observed_min,observed_max,category_levels,n_unique,missing_count,...,quality_limitation,september_handling,meaning_status,purpose,inputs,exact_derivation,allowed_values,field_version,owner,dictionary_version
0,log1p_loss,derived,None,derived target,float64,0.512824,11.703655,None,158223,0,...,Log-transformed values are not dollar amounts ...,Use for exploratory analysis only. Retain orig...,documented,Support exploratory analysis of the loss distr...,loss,log1p_loss = ln(1 + loss),"[0, infinity)",1.0,Liam,1.0.0


## Combined Dictionary

Combine the delivered and team-created fields into one shared dictionary.

In [12]:
data_dictionary = pd.concat(
    [
        source_dictionary,
        derived_dictionary,
    ],
    ignore_index=True
)

print(f"Source fields: {len(source_dictionary)}")
print(f"Derived fields: {len(derived_dictionary)}")
print(f"Total fields: {len(data_dictionary)}")

data_dictionary

Source fields: 132
Derived fields: 1
Total fields: 133


,field_name,field_origin,source_name,approved_role,observed_dtype,observed_min,observed_max,category_levels,n_unique,missing_count,...,quality_limitation,september_handling,meaning_status,purpose,inputs,exact_derivation,allowed_values,field_version,owner,dictionary_version
0,id,source,id,identifier,int64,1.000000,587633.000000,None,188318,0,...,Identifier values have no documented analytica...,Use for integrity checks only. Exclude from di...,documented,None,None,None,None,None,None,1.0.0
1,cat1,source,cat1,anonymous nominal categorical predictor,object,NaN,NaN,"[""A"", ""B""]",2,0,...,Anonymous predictor; semantic meaning is not d...,Treat as an unordered nominal categorical pred...,unresolved,None,None,None,None,None,None,1.0.0
2,cat2,source,cat2,anonymous nominal categorical predictor,object,NaN,NaN,"[""A"", ""B""]",2,0,...,Anonymous predictor; semantic meaning is not d...,Treat as an unordered nominal categorical pred...,unresolved,None,None,None,None,None,None,1.0.0
3,cat3,source,cat3,anonymous nominal categorical predictor,object,NaN,NaN,"[""A"", ""B""]",2,0,...,Anonymous predictor; semantic meaning is not d...,Treat as an unordered nominal categorical pred...,unresolved,None,None,None,None,None,None,1.0.0
4,cat4,source,cat4,anonymous nominal categorical predictor,object,NaN,NaN,"[""A"", ""B""]",2,0,...,Anonymous predictor; semantic meaning is not d...,Treat as an unordered nominal categorical pred...,unresolved,None,None,None,None,None,None,1.0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,cont12,source,cont12,anonymous continuous predictor,float64,0.036232,0.998484,None,328,0,...,Original unit and transformation are not docum...,Treat as a continuous predictor on the deliver...,unresolved,None,None,None,None,None,None,1.0.0
129,cont13,source,cont13,anonymous continuous predictor,float64,0.000228,0.988494,None,353,0,...,Original unit and transformation are not docum...,Treat as a continuous predictor on the deliver...,unresolved,None,None,None,None,None,None,1.0.0
130,cont14,source,cont14,anonymous continuous predictor,float64,0.179722,0.844848,None,18740,0,...,Original unit and transformation are not docum...,Treat as a continuous predictor on the deliver...,unresolved,None,None,None,None,None,None,1.0.0
131,loss,source,loss,regression target,float64,0.670000,121012.250000,None,158223,0,...,Closed claims without payment are excluded fro...,Retain as the regression target. Use original ...,documented,None,None,None,None,None,None,1.0.0


## Dictionary Check

Check that required metadata is present before saving the final artifact.

In [13]:
required_source_columns = [
    "field_name",
    "source_name",
    "approved_role",
    "observed_dtype",
    "n_unique",
    "missing_count",
    "missing_pct",
    "unit_or_scale",
    "documented_meaning",
    "quality_limitation",
    "september_handling",
]

source_missing_metadata = (
    source_dictionary[required_source_columns]
    .isna()
    .sum()
)

source_missing_metadata

field_name            0
source_name           0
approved_role         0
observed_dtype        0
n_unique              0
missing_count         0
missing_pct           0
unit_or_scale         0
documented_meaning    0
quality_limitation    0
september_handling    0
dtype: int64

In [14]:
required_derived_columns = [
    "field_name",
    "approved_role",
    "purpose",
    "inputs",
    "exact_derivation",
    "allowed_values",
    "field_version",
    "owner",
]

derived_missing_metadata = (
    derived_dictionary[required_derived_columns]
    .isna()
    .sum()
)

derived_missing_metadata

field_name          0
approved_role       0
purpose             0
inputs              0
exact_derivation    0
allowed_values      0
field_version       0
owner               0
dtype: int64

In [15]:
source_metadata_ok = (
    source_missing_metadata.sum() == 0
)

derived_metadata_ok = (
    derived_missing_metadata.sum() == 0
)

unique_names = data_dictionary["field_name"].is_unique

print(f"Source metadata complete: {source_metadata_ok}")
print(f"Derived metadata complete: {derived_metadata_ok}")
print(f"Field names unique: {unique_names}")

if not (
    source_metadata_ok
    and derived_metadata_ok
    and unique_names
):
    raise RuntimeError(
        "Data dictionary is incomplete. "
        "Review the checks above before saving."
    )

Source metadata complete: True
Derived metadata complete: True
Field names unique: True


## Unknown Meanings

Anonymous predictor meanings should remain `unknown` unless source documentation provides an approved mapping.

In [16]:
anonymous_fields = source_dictionary[
    source_dictionary["field_name"].isin(
        CAT_COLS + CONT_COLS
    )
]

incorrect_meanings = anonymous_fields[
    anonymous_fields["documented_meaning"] != "unknown"
]

print(
    f"Anonymous predictors marked unknown: "
    f"{len(anonymous_fields) - len(incorrect_meanings)}/"
    f"{len(anonymous_fields)}"
)

incorrect_meanings

Anonymous predictors marked unknown: 130/130


,field_name,field_origin,source_name,approved_role,observed_dtype,observed_min,observed_max,category_levels,n_unique,missing_count,...,quality_limitation,september_handling,meaning_status,purpose,inputs,exact_derivation,allowed_values,field_version,owner,dictionary_version


## Save Data Dictionary

Save the validated dictionary to the shared September deliverables.

In [17]:
data_dictionary.to_csv(
    DICTIONARY_PATH,
    index=False
)

print(
    f"Saved {len(data_dictionary)} fields to "
    f"{DICTIONARY_PATH.relative_to(REPO_ROOT)}"
)

Saved 133 fields to notebooks\final-deliverables\September\Gate 2\data_dictionary.csv


## Result

In [18]:
print("=" * 55)
print("DATA DICTIONARY")
print("=" * 55)
print(f"Dictionary version: {DICTIONARY_VERSION}")
print(f"Source fields:      {len(source_dictionary)}")
print(f"Derived fields:     {len(derived_dictionary)}")
print(f"Total fields:       {len(data_dictionary)}")
print(f"Unknown cat/cont:   {len(anonymous_fields)}")
print("=" * 55)
print("DATA DICTIONARY: PASS")

DATA DICTIONARY
Dictionary version: 1.0.0
Source fields:      132
Derived fields:     1
Total fields:       133
Unknown cat/cont:   130
DATA DICTIONARY: PASS
